In [1]:
%cd /glade/derecho/scratch/lizhili/m2l8/sr_model_code/SRNO_M2L8

/glade/derecho/scratch/lizhili/m2l8/sr_model_code/SRNO_M2L8


/glade/derecho/scratch/lizhili/SRNO/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tensorflow as tf
import numpy as np
import torch.nn as nn

2026-01-25 00:11:57.427534: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # Use only GPU 1

In [4]:
import argparse
import yaml
import os
parser = argparse.ArgumentParser()
parser.add_argument('--config',default='./configs/train_edsr-sronet.yaml')
parser.add_argument('--name', default=None)
parser.add_argument('--tag', default=None)
parser.add_argument('--gpu', default='0,1')
args, _ = parser.parse_known_args()

os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu

with open(args.config, 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)
    print('config loaded.')

config loaded.


In [5]:
import models

model = models.make(config['model']).cuda()

In [6]:
import utils
optimizer = utils.make_optimizer(
            model.parameters(), config['optimizer'])

In [7]:
# from utils import make_coord

# img_lr = torch.randn(1, 7, 64, 64)
# img_hr = torch.randn(1, 7, 256, 256)

# bs = img_lr.shape[0]
# h_hres = img_hr.shape[2]
# w_hres = img_hr.shape[3]

# coord = make_coord([h_hres, w_hres], flatten=False)
# coord = coord.unsqueeze(0).expand(bs, *coord.shape[:2], 2)

# cell = torch.tensor([2 / h_hres, 2 / w_hres], dtype=torch.float32).unsqueeze(0).expand(bs, 2)
# batch = {
#         'inp': img_lr.cuda(),
#         'coord': coord.cuda(),
#         'cell': cell.cuda(),
#         'gt': img_hr.cuda(),
#         }

# pred = model(batch['inp'], batch['coord'], batch['cell'])
# pred.shape

torch.Size([1, 7, 256, 256])

In [7]:
def input_pipeline(filename, batch_size, is_shuffle=True, is_train=True, is_repeat=True):
    feature_description = {
        'lres': tf.io.FixedLenFeature([60*60*7], dtype=tf.int64),
        'hres': tf.io.FixedLenFeature([1000*1000*7], dtype=tf.int64),
    }

    def _parse_function(example_proto):
        feature_dict = tf.io.parse_single_example(example_proto, feature_description)
        lres_img = tf.reshape(feature_dict['lres'], [7, 60, 60])
        hres_img = tf.reshape(feature_dict['hres'], [7, 1000, 1000])

        return lres_img, hres_img

    def _augment_function(lres_img, hres_img):
    # Transpose to [H, W, C]
        lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
        hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]

        # Randomly choose 0, 90, 180, or 270 degrees
        k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)

        # Apply rotation
        lres_img = tf.image.rot90(lres_img, k=k)
        hres_img = tf.image.rot90(hres_img, k=k)

        # Transpose back to [C, H, W]
        lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
        hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]

        return lres_img, hres_img

    dataset = tf.data.TFRecordDataset(filename)
    if is_repeat:
        dataset = dataset.repeat()
    dataset = dataset.map(_parse_function)
    if is_train:
        dataset = dataset.map(_augment_function)
    if is_shuffle:
        dataset = dataset.shuffle(buffer_size=100)
    batch = dataset.batch(batch_size=batch_size)
    return batch

# filenames = ['/content/drive/MyDrive/GeoSR/L8MODIS_30000_2/L8MODIS.tfrecords']
# ds = input_pipeline(filenames, batch_size=5, is_shuffle=False, is_train=True, is_repeat=True)

# for lres_batch, hres_batch in ds:
#     print(lres_batch.shape)
#     print(hres_batch.shape)

#     for i in range(lres_batch.shape[0]):
#         fig, axes = plt.subplots(1, 2, figsize=(8, 4))

#         lres_img = np.transpose(lres_batch[i].numpy(), (1, 2, 0))/3500.0
#         hres_img = np.transpose(hres_batch[i].numpy(), (1, 2, 0))/255.0

#         axes[0].imshow(lres_img[:, :, 3:0:-1])
#         axes[0].set_title('Low-Resolution')
#         axes[0].axis('off')

#         axes[1].imshow(hres_img[:, :, :3])
#         axes[1].set_title('High-Resolution')
#         axes[1].axis('off')

#         plt.show()

#     break


In [ ]:
def load_matched_weights(model, checkpoint_path):
    # Load checkpoint (state_dict)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
    # state_dict = state_dict['params']

    # Filter only matching keys
    model_dict = model.state_dict()
    # print(state_dict['params'].keys())
    # print(model_dict.keys())
    matched_dict = {k: v for k, v in state_dict.items() if k in model_dict and v.shape == model_dict[k].shape}

    # Load the matched weights
    model_dict.update(matched_dict)
    model.load_state_dict(model_dict)

    print(f"✅ Loaded {len(matched_dict)} matching parameters out of {len(model_dict)} total.")

    return model

model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/SR_finetuned_weights/SRNO_M2L8_x4_weights_new.pth')
# model = load_matched_weights(model, '/glade/derecho/scratch/lizhili/m2l8/SRNO_M2L8_x16_weights_new.pth')

In [9]:
def random_crop_lr_hr(img_lr, img_hr, lr_crop_size):
    """
    Random aligned crop for super-resolution pairs.

    img_lr: Tensor [B, C, 64, 64]
    img_hr: Tensor [B, C, 1024, 1024]
    lr_crop_size: int (e.g., 32)

    Returns:
        lr_crop: [B, C, lr_crop_size, lr_crop_size]
        hr_crop: [B, C, lr_crop_size*16, lr_crop_size*16]
    """
    scale = img_hr.shape[-1] // img_lr.shape[-1]  # 16

    _, _, H_lr, W_lr = img_lr.shape
    assert H_lr >= lr_crop_size and W_lr >= lr_crop_size

    top_lr = torch.randint(0, H_lr - lr_crop_size + 1, (1,)).item()
    left_lr = torch.randint(0, W_lr - lr_crop_size + 1, (1,)).item()

    top_hr = top_lr * scale
    left_hr = left_lr * scale

    lr_crop = img_lr[:, :, top_lr:top_lr+lr_crop_size,
                             left_lr:left_lr+lr_crop_size]

    hr_crop = img_hr[:, :, top_hr:top_hr+lr_crop_size*scale,
                             left_hr:left_hr+lr_crop_size*scale]

    return lr_crop, hr_crop

In [ ]:
import torch.nn.functional as F
import tensorflow as tf
import torch
from utils import make_coord

filenames = ['/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_0.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_1.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_2.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_3.tfrecords',
             '/glade/derecho/scratch/lizhili/m2l8/sr_ds/M2L8_SR_part_4.tfrecords']
ds = input_pipeline(filenames, batch_size=2, is_shuffle=True, is_train=True, is_repeat=False)
model.train()
loss_fn = torch.nn.L1Loss()



for epoch in range(18):
    print(f'Epoch {epoch}')

    scaler = torch.cuda.amp.GradScaler()
    train_loss = utils.Averager()

    for step, (lr, hr) in enumerate(ds):

        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2
        # img_lr = F.interpolate(lr, size=(64, 64), mode='nearest')
        # img_hr = F.interpolate(hr, size=(1024, 1024), mode='nearest')
        img_lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        img_hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
        img_lr, img_hr = random_crop_lr_hr(img_lr, img_hr, lr_crop_size=32)
        if step == 0:
            print(img_lr.shape)
            print(img_hr.shape)


        bs = img_lr.shape[0]
        h_hres = img_hr.shape[2]
        w_hres = img_hr.shape[3]

        coord = make_coord([h_hres, w_hres], flatten=False)
        coord = coord.unsqueeze(0).expand(bs, *coord.shape[:2], 2)

        cell = torch.tensor([2 / h_hres, 2 / w_hres], dtype=torch.float32).unsqueeze(0).expand(bs, 2)
        batch = {
                'inp': img_lr.cuda(),
                'coord': coord.cuda(),
                'cell': cell.cuda(),
                'gt': img_hr.cuda(),
                }

        pred = model(batch['inp'], batch['coord'], batch['cell'])

        loss = loss_fn(pred, batch['gt'])

        current_iter = step+1
        train_loss.add(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 500 == 0:
            print(f'Step {step}, Loss: {loss.item()}')

    print(f'Epoch {epoch}, Loss: {train_loss.item()}')
    torch.save(model.state_dict(), '/glade/derecho/scratch/lizhili/m2l8/SRNO_M2L8_x16_weights_new.pth')

In [11]:
from utils import make_coord

def downstream_run(lres_size ,
                    hres_size ,
                    hres_size_4x ,
                    label_size ,
                    num_sample,
                    num_training ,
                    finetune_tfrecords,
                    finetuned_model,
                    output_tfrecords,
                    model):
    optimizer = utils.make_optimizer(model.parameters(), config['optimizer'])
    model.load_state_dict(torch.load('/glade/derecho/scratch/lizhili/m2l8/SRNO_M2L8_x16_weights_new.pth', weights_only=True))
    num_test = num_sample-num_training


    def input_pipeline_downstream_sr(filename, batch_size, skip, take, is_shuffle=True, is_train=True, is_repeat=True):
        feature_description = {
            'lres': tf.io.FixedLenFeature([7*lres_size*lres_size], dtype=tf.int64),
            'hres': tf.io.FixedLenFeature([7*hres_size*hres_size], dtype=tf.int64),
            'label': tf.io.FixedLenFeature([label_size*label_size], dtype=tf.int64)
        }

        @tf.function
        def _parse_function(example_proto):
            feature_dict = tf.io.parse_single_example(example_proto, feature_description)

            lres = feature_dict['lres']
            lres = tf.reshape(lres, [7, lres_size, lres_size])
            lres = tf.cast(lres, tf.float32)

            hres = feature_dict['hres']
            hres = tf.reshape(hres, [7, hres_size, hres_size])
            hres = tf.cast(hres, tf.float32)

            label = feature_dict['label']
            label = tf.reshape(label, [label_size, label_size, 1])
            return lres, hres, label

        @tf.function
        def _augment_function(lres_img, hres_img, label):
        # Transpose to [H, W, C]
            lres_img = tf.transpose(lres_img, [1, 2, 0])   # [60, 60, 12]
            hres_img = tf.transpose(hres_img, [1, 2, 0])   # [1000, 1000, 4]
    
            # Randomly choose 0, 90, 180, or 270 degrees
            k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    
            # Apply rotation
            lres_img = tf.image.rot90(lres_img, k=k)
            hres_img = tf.image.rot90(hres_img, k=k)
            label = tf.image.rot90(label, k=k)
    
            # Transpose back to [C, H, W]
            lres_img = tf.transpose(lres_img, [2, 0, 1])   # [12, 60, 60]
            hres_img = tf.transpose(hres_img, [2, 0, 1])   # [4, 1000, 1000]
    
            return lres_img, hres_img, label

        dataset = tf.data.TFRecordDataset(filename)
        dataset = dataset.skip(skip)
        if take:
            dataset = dataset.take(take)
        if is_repeat:
            dataset = dataset.repeat()
        dataset = dataset.map(_parse_function)
        if is_train:
            dataset = dataset.map(_augment_function)
        if is_shuffle:
            dataset = dataset.shuffle(buffer_size=100)
        batch = dataset.batch(batch_size=batch_size)

        return batch


    # --------------------------------------------
    print('Begin Finetune')
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 2, 0, num_training, is_shuffle=True, is_train=True, is_repeat=False)

    for epoch in range(10):
        print(f'Epoch {epoch}')

        for step, (lr, hr, _) in enumerate(ds):
            lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
            hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2

            img_lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
            img_hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)
            img_lr, img_hr = random_crop_lr_hr(img_lr, img_hr, lr_crop_size=32)

            bs = img_lr.shape[0]
            h_hres = img_hr.shape[2]
            w_hres = img_hr.shape[3]

            coord = make_coord([h_hres, w_hres], flatten=False)
            coord = coord.unsqueeze(0).expand(bs, *coord.shape[:2], 2)

            cell = torch.tensor([2 / h_hres, 2 / w_hres], dtype=torch.float32).unsqueeze(0).expand(bs, 2)
            batch = {
                    'inp': img_lr.cuda(),
                    'coord': coord.cuda(),
                    'cell': cell.cuda(),
                    'gt': img_hr.cuda(),
                    }

            pred = model(batch['inp'], batch['coord'], batch['cell'])

            loss = torch.nn.L1Loss()(pred, batch['gt'])

            current_iter = step+1

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if step % 500 == 0:
                print(f'Step {step}, Loss: {loss.item()}')

        torch.save(model.state_dict(), finetuned_model)

    # --------------------------------------------

    print('Begin Write SR dataset')
    # TFRecord writer setup
    writer = tf.io.TFRecordWriter(output_tfrecords)

    # Serialization function
    def serialize_example(hres, label):
        feature = {
            'hres': tf.train.Feature(int64_list=tf.train.Int64List(value=hres.reshape(-1))),
            'label': tf.train.Feature(int64_list=tf.train.Int64List(value=label.reshape(-1)))
        }
        example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
        return example_proto.SerializeToString()

    # Inference loop
    ds = input_pipeline_downstream_sr(finetune_tfrecords, 1, 0, num_sample, is_shuffle=False, is_train=False, is_repeat=False)
    for step, (lr, hr, label) in enumerate(ds):
        lr = torch.from_numpy(lr.numpy().astype('float32')).to('cuda')*0.0001
        hr = torch.from_numpy(hr.numpy().astype('float32')).to('cuda')*0.0000275-0.2

        img_lr = F.interpolate(lr, size=(64, 64), mode='bilinear', align_corners=False)
        img_hr = F.interpolate(hr, size=(1024, 1024), mode='bilinear', align_corners=False)

        bs = img_lr.shape[0]
        h_hres = img_hr.shape[2]
        w_hres = img_hr.shape[3]

        coord = make_coord([h_hres, w_hres], flatten=False)
        coord = coord.unsqueeze(0).expand(bs, *coord.shape[:2], 2)

        cell = torch.tensor([2 / h_hres, 2 / w_hres], dtype=torch.float32).unsqueeze(0).expand(bs, 2)
        batch = {
                'inp': img_lr.cuda(),
                'coord': coord.cuda(),
                'cell': cell.cuda(),
                'gt': img_hr.cuda(),
                }
        with torch.no_grad():
            output = model(batch['inp'], batch['coord'], batch['cell'])
        output = output.detach().cpu().numpy()
        output = np.clip(output, 0, 1)
        output = ((output+0.2)/0.0000275).astype(int)
        output[output<0]=0
        label = label.numpy()

        if step == 0:
            lr = lr.detach().cpu().numpy()

        for i in range(output.shape[0]):
            print(output[i].shape)
            print(label[i].shape)
            example = serialize_example(output[i], label[i])
            writer.write(example)

            if step == 0:
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                lr_show = np.transpose(lr[i], (1, 2, 0))
                axes[0].imshow(2*lr_show[:, :, [0,3,2]])
                axes[0].set_title('Low-Resolution')
                axes[0].axis('off')

                output_show = np.transpose(output[i], (1, 2, 0))
                axes[1].imshow(2*(output_show[:, :, 3:0:-1]*0.0000275-0.2))
                axes[1].set_title('High-Resolution')
                axes[1].axis('off')

                axes[2].imshow(label[i])
                axes[2].set_title('Output')
                axes[2].axis('off')

                plt.show()

    # Close writer
    writer.close()

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1065
num_training = 852
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_River.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/SRNO_x16_M2L8_River_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_River_SRNO_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 762
num_training = 610
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CDL.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/SRNO_x16_M2L8_CDL_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CDL_SRNO_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 1687
num_training = 1350
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_Urban.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/SRNO_x16_M2L8_Urban_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_Urban_SRNO_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)



In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 1024  # 4x sr image size
label_size = 1000 #Vermontlc
num_sample = 755
num_training = 604
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_GPP.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/SRNO_x16_M2L8_GPP_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_GPP_SRNO_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)

In [ ]:
lres_size = 60
hres_size = 1000
hres_size_4x = 256  # 4x sr image size
label_size = 1000 #Vermontlc
class_num = 1
num_sample = 1408
num_training = 1126
finetune_tfrecords = ['/glade/derecho/scratch/lizhili/m2l8/down_ds/M2L8_CHM.tfrecords']
finetuned_model = '/glade/derecho/scratch/lizhili/m2l8/SRNO_x16_M2L8_CHM_Finetune.pth'
output_tfrecords = '/glade/derecho/scratch/lizhili/m2l8/M2L8_CHM_SRNO_x16.tfrecords'

downstream_run(lres_size ,
                hres_size ,
                hres_size_4x ,
                label_size ,
                num_sample,
                num_training ,
                finetune_tfrecords,
                finetuned_model,
                output_tfrecords,
                model)